# SNR-opt merging with N-tuple collocation

Compare weighted averaging (WA), SNR-opt, maxR, and equal weighting (EW)
on collocated synthetic observations with correlated errors. The estimated
workflow uses one NC fit for the error covariance, loadings, and product SNRs.

- SNR-opt. [Kim et al. (2022)](https://doi.org/10.1109/TGRS.2021.3107028)
- N-tuple collocation. [Pham and Kim (2027)](https://doi.org/10.1016/j.jhydrol.2026.136749)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from snropt_helpers import data_gen, evaluate_metrics, maxR, snr_est, snr_opt, wa

## Step 1: Data parameters

In [ ]:
p = 3
n = 100
np.random.seed(123)
a = np.random.rand(p)
ecc = 0.3   # common error cross-correlation
SNRdB = 0.1 # signal-to-average-error ratio used by the generator

## Step 2: Synthetic data generation

In [ ]:
# Signal and error
y, e = data_gen(n, p, ecc, SNRdB)

# Observation: x = a*signal + error
x = y[:, None] * a[None, :] + e

# Signal power and covariance matrices
Ey2 = np.var(y, ddof=1)
EeeT = np.cov(e, rowvar=False)
ExxT = np.cov(x, rowvar=False)
N = EeeT / Ey2

# Original dataset performance
U_eye = np.eye(p)
MSE_ori, R2_ori = evaluate_metrics(ExxT, Ey2, a, U_eye)

print('+ Metrics for original data')
print(' * MSE for x:', np.round(MSE_ori, 3))
print(' * R2 for x: ', np.round(R2_ori, 3))


## Step 3. Merging using true parameters

In [ ]:
ue = np.ones(p) / p
methods = ("WA", "SNRopt", "maxR", "EW")
U_true = np.column_stack((wa(EeeT), snr_opt(N, a), maxR(a, ExxT), ue))
MSE_true, R2_true = evaluate_metrics(ExxT, Ey2, a, U_true)

def print_metrics(mse, r2):
    print(f"{'Method':<8} {'MSE':>10} {'R^2':>10}")
    for name, error, skill in zip(methods, mse, r2):
        print(f"{name:<8} {error:10.6f} {skill:10.6f}")

print_metrics(MSE_true, R2_true)

## Step 4. Merging using one NC fit

The example uses half of the simulated signal variance as a deliberately
rough proxy for a target variance supplied from outside the covariance fit.
In a reference-free application without a target variance, omit that argument
to obtain an estimate on a unit-variance latent target scale. The per-product
SNRs and estimated error covariance are independent of this choice.

In [ ]:
Ey2_est = 0.5 * Ey2
N_est, a_est, snr, fit = snr_est(ExxT, Ey2_est, return_fit=True)
U_est = np.column_stack((
    wa(fit["errorCov"], fit["info"]["covarianceScale"]), snr_opt(N_est, a_est),
    maxR(fit["theta"], fit["covariance"]), ue,
))
MSE_est, R2_est = evaluate_metrics(ExxT, Ey2, a, U_est)

print("NC linear SNR by product:", np.round(snr, 4))
print(f"NC minimum error covariance eigenvalue: {fit['info']['minErrorCovEigenvalue']:.6g}")
print_metrics(MSE_est, R2_est)

## Step 5. Plot MSE and squared correlation

In [ ]:
xpos = np.arange(len(methods))
width = 0.25
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")

for ax, label, original, ideal, estimated in (
    (axes[0], "MSE", MSE_ori, MSE_true, MSE_est),
    (axes[1], r"$R^2$", R2_ori, R2_true, R2_est),
):
    ax.bar(xpos - width, np.full(len(methods), original.mean()), width,
           label="Mean individual product")
    ax.bar(xpos, ideal, width, label="True parameters")
    ax.bar(xpos + width, estimated, width, label="NC estimates")
    ax.set(ylabel=label, xticks=xpos, xticklabels=methods)

axes[0].legend()
plt.show()